# PM4Py Process Mining on UNRDF OTEL Logs

This notebook extracts event logs from Loki and applies process mining techniques:
1. **Log Extraction** — Query Loki for OTEL span events
2. **Event Log Construction** — Convert traces to process mining format
3. **Process Discovery** — Discover BPMN and Petri net models
4. **Conformance Checking** — Compare actual vs expected behavior
5. **Bottleneck Analysis** — Identify performance bottlenecks

In [ ]:
# Papermill parameters (overridable via CLI)
# Usage: papermill pm4py-process-mining.ipynb output.ipynb -p hours 6 -p limit 500
hours = 24  # Time range in hours to query
limit = 1000  # Max traces/logs to fetch
service_filter = ""  # Filter by service name (empty = all)

import pm4py
import requests
import pandas as pd
import json
import re
from datetime import datetime, timedelta
import os

LOKI_URL = os.environ.get("LOKI_URL", "http://loki:3100")
TEMPO_URL = os.environ.get("TEMPO_URL", "http://tempo:3200")
PROMETHEUS_URL = os.environ.get("PROMETHEUS_URL", "http://prometheus:9090")

## 1. Extract Traces from Tempo
Query Tempo API for traces and convert to event log format.

In [ ]:
def fetch_traces_from_tempo(hours=1, limit=1000):
    """Fetch traces from Tempo and return as DataFrame."""
    end = int(datetime.utcnow().timestamp() * 1e9)
    start = int((datetime.utcnow() - timedelta(hours=hours)).timestamp() * 1e9)
    
    # Search for all traces in the time range
    url = f"{TEMPO_URL}/api/search?start={start}&end={end}&limit={limit}"
    resp = requests.get(url)
    resp.raise_for_status()
    
    traces = resp.json().get('traces', [])
    events = []
    
    for trace in traces:
        trace_id = trace['traceID']
        
        # Fetch full trace
        trace_url = f"{TEMPO_URL}/api/traces/{trace_id}"
        trace_resp = requests.get(trace_url)
        if trace_resp.status_code != 200:
            continue
        
        trace_data = trace_resp.json()
        for batch in trace_data.get('batches', []):
            for scope_span in batch.get('scopeSpans', []):
                for span in scope_span.get('spans', []):
                    attrs = {}
                    for a in span.get('attributes', []):
                        v = a.get('value', {})
                        attrs[a['key']] = v.get('stringValue') or v.get('intValue') or str(v)
                    
                    events.append({
                        'case:concept:name': trace_id,
                        'concept:name': span['name'],
                        'time:timestamp': datetime.fromtimestamp(
                            int(span['startTimeUnixNano']) / 1e9
                        ).isoformat(),
                        'org:resource': attrs.get('service.name', 'unknown'),
                        'duration_ms': (int(span.get('endTimeUnixNano', 0)) - int(span.get('startTimeUnixNano', 0))) / 1e6,
                        'span_id': span['spanId'],
                        'mcp.tool.name': attrs.get('mcp.tool.name', ''),
                        'mcp.tool.success': attrs.get('mcp.tool.success', ''),
                    })
    
    return pd.DataFrame(events)

df = fetch_traces_from_tempo(hours=24)
print(f"Extracted {len(df)} events from {df['case:concept:name'].nunique()} traces")
df.head(10)

## 2. Extract Logs from Loki
Query Loki for container logs and convert to event log format.

In [ ]:
def fetch_logs_from_loki(hours=1, limit=1000):
    """Fetch logs from Loki and return as DataFrame."""
    end = int(datetime.utcnow().timestamp())
    start = int((datetime.utcnow() - timedelta(hours=hours)).timestamp())
    
    query = '{container=~"unrdf.*"}'
    url = f"{LOKI_URL}/loki/api/v1/query_range"
    params = {
        'query': query,
        'start': str(start) + '000000000',
        'end': str(end) + '000000000',
        'limit': limit,
        'direction': 'forward',
    }
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    
    data = resp.json()
    events = []
    
    for result in data.get('data', {}).get('result', []):
        container = result.get('stream', {}).get('container', 'unknown')
        for value in result.get('values', []):
            ts_ns, line = value
            ts = datetime.fromtimestamp(int(ts_ns) / 1e9).isoformat()
            
            # Extract trace ID if present
            trace_id = ''
            if '"traceID"' in line:
                import re
                m = re.search(r'"traceID":"([a-f0-9]+)"', line)
                if m:
                    trace_id = m.group(1)
            
            events.append({
                'case:concept:name': trace_id or container,
                'concept:name': 'log_entry',
                'time:timestamp': ts,
                'org:resource': container,
                'log_level': 'info' if 'info' in line.lower() else ('error' if 'error' in line.lower() else 'debug'),
                'message': line[:200],
            })
    
    return pd.DataFrame(events)

log_df = fetch_logs_from_loki(hours=24)
print(f"Extracted {len(log_df)} log entries")
log_df.head(10)

## 3. Process Discovery — Inductive Miner
Discover a process model from the event log.

In [ ]:
# Combine trace events and log events
if len(df) > 0:
    # Convert timestamp
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])
    
    # Discover process tree using Inductive Miner
    net, im, fm = pm4py.discover_petri_net_inductive(df)
    print(f"Places: {len(net.places)}, Transitions: {len(net.transitions)}, Arcs: {len(net.arcs)}")
    
    # Visualize
    pm4py.view_petri_net(net, im, fm)
else:
    print("No trace events found. Generate traces first using the HotROD demo app.")
    print("Open http://localhost:8080 and click around to generate traces.")

## 4. BPMN Process Model Discovery

In [ ]:
if len(df) > 0:
    bpmn_model = pm4py.discover_bpmn_inductive(df)
    pm4py.view_bpmn(bpmn_model)
else:
    print("No events to discover model from.")

## 5. Conformance Checking
Check how well traces conform to the discovered model.

In [ ]:
if len(df) > 0:
    # Token-based replay fitness
    fitness = pm4py.fitness_token_based_replay(df, net, im, fm)
    print(f"Fitness: {fitness}")
    
    # Precision
    precision = pm4py.precision_token_based_replay(df, net, im, fm)
    print(f"Precision: {precision}")
else:
    print("No events for conformance checking.")

## 6. Bottleneck Analysis
Identify the slowest activities in the process.

In [ ]:
if len(df) > 0 and 'duration_ms' in df.columns:
    # Activity duration analysis
    activity_stats = df.groupby('concept:name')['duration_ms'].agg(['mean', 'median', 'max', 'count'])
    activity_stats = activity_stats.sort_values('mean', ascending=False)
    print("Activity Duration Analysis (ms):")
    print(activity_stats.to_string())
    
    # Case duration
    case_duration = df.groupby('case:concept:name').agg(
        start=('time:timestamp', 'min'),
        end=('time:timestamp', 'max'),
        span_count=('concept:name', 'count')
    )
    case_duration['duration'] = (case_duration['end'] - case_duration['start']).dt.total_seconds()
    print(f"\nCase Duration: mean={case_duration['duration'].mean():.2f}s, max={case_duration['duration'].max():.2f}s")
else:
    print("No events for bottleneck analysis.")

## 7. Process Map Visualization
Visualize the process flow as a frequency map.

In [ ]:
if len(df) > 0:
    pm4py.view_process_tree(pm4py.discover_process_tree_inductive(df))
else:
    print("No events for visualization.")